In [1]:
import requests
import json
import datetime
import tqdm
import time
import os
import shutil

In [2]:
api_url = 'https://api.jikan.moe/v4'

def scrape_page(endpoint, page, file_path):
    response = requests.get(api_url + endpoint + f'?page={page}')
    response.raise_for_status()
    data = response.json()
    with open(file_path, 'w') as f:
        json.dump(data['data'], f, indent=4)

In [3]:
wait = 1.2 # seconds, with 1.15 crashed

def scrape_jikan_db(database):

    directory_path = f'data/raw/{database}'
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
    
    last_page = requests.get(api_url + '/' + database).json()['pagination']['last_visible_page']
    length = len(str(last_page))

    print('Started:', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    
    for page in tqdm.trange(1, last_page + 1):
        start = time.perf_counter()
        scrape_page('/' + database, page, directory_path + f'/page{str(page).zfill(length)}.json')
        end = time.perf_counter()
        time.sleep(max(0, start + wait - end))
    
    print('Finished:', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
def merge_files(database):

    directory_path = f'data/raw/{database}'

    data = []
    for file_name in tqdm.tqdm(os.listdir(directory_path)):
        file_path = os.path.join(directory_path, file_name)
        with open(file_path, 'r') as f:
            file = json.load(f)
        data.extend(file)
    
    with open(f'data/raw/{database}.json', 'w') as f:
        json.dump(data, f, indent=4)
    
    shutil.rmtree(directory_path)

In [ ]:
scrape_jikan_db('manga')

In [5]:
merge_files('manga')

100%|██████████████████████████████████████| 3227/3227 [00:03<00:00, 923.97it/s]
